In [ ]:

import os
import json
import re
from typing import Dict, Any, Tuple
from groq import Groq

# ---------- Configuration ----------
MODEL_NAME = "llama-3.1-70b-versatile"   # Good general-purpose extractor
TEMPERATURE = 0.0                        # Deterministic for extraction
MAX_TOKENS = 1024

# ---------- Prompt (Generic + Fallback + Sanity) ----------
SYSTEM_PROMPT = """You are a meticulous information extraction assistant.
You extract structured data from US IRS Form W-2 text (OCR or plain text).
Return ONLY a valid JSON object (no prose).
If uncertain, set the value to null and add a reason to the 'errors' list.
Adhere to the schema exactly.

Sanity & Validation Rules:
- w2_year: 4-digit integer; must be between 1990 and 2035.
- employee_ssn: format 'XXX-XX-XXXX'. If not found, set to null. Do NOT fabricate.
- employer_name, employer_address, employee_name, employee_address:
  - Prefer values near canonical labels (Employer’s name, Employer’s address, Employee’s name, etc.).
  - Normalize whitespace and remove leading/trailing punctuation.
- wages_tips_other_comp: positive number (Box 1). Strip $ and commas; cast to float. If multiple candidates, pick the one nearest "Box 1", "Wages, tips, other compensation", or "1.".
- medicare_wages_tips: positive number (Box 5). Strip $ and commas; cast to float. Prefer nearest "Box 5" or "Medicare wages and tips".
- If both Box 1 and Box 5 appear multiple times, pick the value in the same row or block as their label.
- Include 'source_spans' with the exact substring(s) used for each field when available.

Output Schema (keys MUST match exactly):
{
  "w2_year": 2024,
  "employee_ssn": "123-45-6789",
  "employer_name": "string or null",
  "employer_address": "string or null",
  "employee_name": "string or null",
  "employee_address": "string or null",
  "wages_tips_other_comp": 50000.00,
  "medicare_wages_tips": 45000.00,
  "source_spans": {
    "w2_year": "substring or null",
    "employee_ssn": "substring or null",
    "employer_name": "substring or null",
    "employer_address": "substring or null",
    "employee_name": "substring or null",
    "employee_address": "substring or null",
    "wages_tips_other_comp": "substring or null",
    "medicare_wages_tips": "substring or null"
  },
  "errors": []
}

If any required field is missing or ambiguous:
- Set value to null,
- Add a concise reason to 'errors' (e.g., "Box 1 label present but value unreadable").

Return JSON only.
"""

def user_prompt_from_text(w2_text: str) -> str:
    return f"""Extract the following fields from this W-2 text:

FIELDS TO EXTRACT
- w2_year
- employee_ssn
- employer_name
- employer_address
- employee_name
- employee_address
- wages_tips_other_comp (Box 1)
- medicare_wages_tips (Box 5)

TEXT
----
{w2_text}
----
"""

# ---------- Groq Client ----------
def get_groq_client() -> Groq:
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("GROQ_API_KEY environment variable is not set.")
    return Groq(api_key=api_key)

# ---------- Sanity Checks ----------
def sanitize_result(data: Dict[str, Any]) -> Tuple[Dict[str, Any], list]:
    errors = data.get("errors", [])
    current_year = 2025  # Adjust if needed for your runtime context

    # Year: 1990–2035
    year = data.get("w2_year")
    if isinstance(year, int):
        if not (1990 <= year <= 2035):
            errors.append(f"w2_year out of expected range: {year}")
            data["w2_year"] = None
    else:
        if year is not None:
            errors.append("w2_year must be integer or null")

    # SSN format
    ssn = data.get("employee_ssn")
    if ssn is not None:
        if not re.fullmatch(r"\d{3}-\d{2}-\d{4}", ssn):
            errors.append(f"employee_ssn invalid format: {ssn}")
            data["employee_ssn"] = None

    # Normalize strings
    for k in ["employer_name", "employer_address", "employee_name", "employee_address"]:
        v = data.get(k)
        if isinstance(v, str):
            nv = " ".join(v.split()).strip(" ,.;:|")
            data[k] = nv if nv else None

    # Monetary fields: positive floats, reasonable upper bound
    def _money_check(key: str):
        val = data.get(key)
        if val is None:
            return
        try:
            f = float(val)
            if f <= 0:
                errors.append(f"{key} must be positive: {f}")
                data[key] = None
            elif f > 50_000_000:  # arbitrary upper bound to catch OCR errors
                errors.append(f"{key} unusually large: {f}")
                data[key] = None
            else:
                data[key] = round(f, 2)
        except Exception:
            errors.append(f"{key} not a number: {val}")
            data[key] = None

    _money_check("wages_tips_other_comp")
    _money_check("medicare_wages_tips")

    data["errors"] = errors
    return data, errors



# ---------- Main Extraction ----------
def extract_w2(w2_text: str) -> Dict[str, Any]:
    client = get_groq_client()

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        response_format={"type": "json_object"},  # Enforce JSON output
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt_from_text(w2_text)}
        ]
    )

    raw = completion.choices[0].message.content

    # Parse JSON from the model
    try:
        data = json.loads(raw)
    except Exception as e:
        # If model returns invalid JSON, use minimal structure
        data = {
            "w2_year": None,
            "employee_ssn": None,
            "employer_name": None,
            "employer_address": None,
            "employee_name": None,
            "employee_address": None,
            "wages_tips_other_comp": None,
            "medicare_wages_tips": None,
            "source_spans": {
                "w2_year": None,
                "employee_ssn": None,
                "employer_name": None,
                "employer_address": None,
                "employee_name": None,
                "employee_address": None,
                "wages_tips_other_comp": None,
                "medicare_wages_tips": None
            },
            "errors": [f"Model returned non-JSON: {e}"]
        }

    # Sanitize / validate
    data, errors = sanitize_result(data)


    return data

# ---------- Example Usage ----------
if __name__ == "__main__":
    # Replace with your OCR text of the W-2
    sample_text = """
    Form W-2 Wage and Tax Statement 2024
    Employee's social security number 123-45-6789
    Employer name: ACME Corp
    Employer address: 123 Main Street, Cleveland, OH 44101
    Employee name: Jane Doe
    Employee address: 456 Elm Ave, Cleveland, OH 44102

    1 Wages, tips, other compensation $50,000.00
    5 Medicare wages and tips $45,000.00
    """

    result = extract_w2(sample_text)
    print(json.dumps(result, indent=2))
